In [1]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine


In [5]:
nav=pd.read_csv(r"C:\Users\Hrugved\OneDrive\Desktop\bluestock\data\raw\02_nav_history.csv")
transactions=pd.read_csv(r"C:\Users\Hrugved\OneDrive\Desktop\bluestock\data\raw\08_investor_transactions.csv")
performance=pd.read_csv(r"C:\Users\Hrugved\OneDrive\Desktop\bluestock\data\raw\07_scheme_performance.csv")

In [ ]:
nav["date"]=pd.to_datetime(nav["date"],errors="coerce")
nav=nav.drop_duplicates()
nav=nav.sort_values(["amfi_code","date"])
nav["nav"]=nav.groupby("amfi_code")["nav"].ffill()
nav=nav[nav["nav"]>0]
nav.to_csv(r"C:\Users\Hrugved\OneDrive\Desktop\bluestock\data\processed\nav_history_clean.csv", index=False)

In [10]:
nav.head()

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [13]:
transactions=pd.read_csv(r"C:\Users\Hrugved\OneDrive\Desktop\bluestock\data\raw\08_investor_transactions.csv")

transactions["transaction_date"]=pd.to_datetime(transactions["transaction_date"],errors="coerce")

transactions["transaction_type"]=transactions["transaction_type"].astype(str).str.strip().replace({
    "sip":"SIP",
    "Sip":"SIP",
    "SIP":"SIP",
    "lumpsum":"Lumpsum",
    "Lumpsum":"Lumpsum",
    "lump sum":"Lumpsum",
    "Lump Sum":"Lumpsum",
    "redeem":"Redemption",
    "Redeem":"Redemption",
    "redemption":"Redemption",
    "Redemption":"Redemption"
})

transactions=transactions[transactions["amount_inr"]>0]

valid_kyc=["Verified","Pending","Rejected"]
invalid_kyc=transactions[~transactions["kyc_status"].isin(valid_kyc)]

transactions=transactions.drop_duplicates()

transactions.to_csv(r"C:\Users\Hrugved\OneDrive\Desktop\bluestock\data\processed\investor_transactions_clean.csv", index=False)

print("Rows:",len(transactions))
print("Duplicates:",transactions.duplicated().sum())
print("Invalid KYC Records:",len(invalid_kyc))
print(transactions.isnull().sum())

Rows: 32778
Duplicates: 0
Invalid KYC Records: 0
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64


In [16]:
performance=pd.read_csv(r"C:\Users\Hrugved\OneDrive\Desktop\bluestock\data\raw\07_scheme_performance.csv")
return_cols=["return_1yr_pct","return_3yr_pct","return_5yr_pct"]
for col in return_cols:
    performance[col]=pd.to_numeric(performance[col],errors="coerce")
anomalies=performance[(performance["return_1yr_pct"]>200)|(performance["return_1yr_pct"]<-100)|(performance["return_3yr_pct"]>500)|(performance["return_3yr_pct"]<-100)|(performance["return_5yr_pct"]>1000)|(performance["return_5yr_pct"]<-100)]
expense_anomalies=performance[(performance["expense_ratio_pct"]<0.1)|(performance["expense_ratio_pct"]>2.5)]
performance=performance.drop_duplicates()
performance.to_csv(r"C:\Users\Hrugved\OneDrive\Desktop\bluestock\data\processed\scheme_performance_clean.csv", index=False)
print("Rows:",len(performance))
print("Return anomalies:",len(anomalies))
print("Expense ratio anomalies:",len(expense_anomalies))
print(performance.isnull().sum())
performance.to_csv(r"C:\Users\Hrugved\OneDrive\Desktop\bluestock\data\processed\scheme_performance_clean.csv", index=False)

Rows: 40
Return anomalies: 0
Expense ratio anomalies: 0
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64


In [17]:
from sqlalchemy import create_engine
import pandas as pd

engine=create_engine("sqlite:///bluestock_mf.db")

In [19]:
nav_clean=pd.read_csv(r"C:\Users\Hrugved\OneDrive\Desktop\bluestock\data\processed\nav_history_clean.csv")
transactions_clean=pd.read_csv(r"C:\Users\Hrugved\OneDrive\Desktop\bluestock\data\processed\investor_transactions_clean.csv")
performance_clean=pd.read_csv(r"C:\Users\Hrugved\OneDrive\Desktop\bluestock\data\processed\scheme_performance_clean.csv")

In [20]:
fund_dim=performance_clean[["amfi_code","scheme_name","fund_house","category"]].drop_duplicates()

In [21]:
fund_dim.to_sql("dim_fund",engine,if_exists="replace",index=False)
nav_clean.to_sql("fact_nav",engine,if_exists="replace",index=False)
transactions_clean.to_sql("fact_transactions",engine,if_exists="replace",index=False)
performance_clean.to_sql("fact_performance",engine,if_exists="replace",index=False)

40

In [22]:
import sqlite3

conn=sqlite3.connect("bluestock_mf.db")

print(pd.read_sql("SELECT COUNT(*) AS rows FROM dim_fund",conn))
print(pd.read_sql("SELECT COUNT(*) AS rows FROM fact_nav",conn))
print(pd.read_sql("SELECT COUNT(*) AS rows FROM fact_transactions",conn))
print(pd.read_sql("SELECT COUNT(*) AS rows FROM fact_performance",conn))

   rows
0    40
    rows
0  46000
    rows
0  32778
   rows
0    40
